# U-Net - TensorFlow / Keras



In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 1. Synthetic Segmentation Data
# -----------------------------
def generate_synthetic_data(num_samples=1000, img_size=128):
    """Generate synthetic images with circular masks for segmentation."""
    images = np.zeros((num_samples, img_size, img_size, 3), dtype=np.float32)
    masks = np.zeros((num_samples, img_size, img_size, 1), dtype=np.float32)

    for i in range(num_samples):
        # Random background
        for c in range(3):
            images[i, :, :, c] = np.random.uniform(0.0, 0.3)

        # Random circle
        cx = np.random.randint(30, img_size - 30)
        cy = np.random.randint(30, img_size - 30)
        r = np.random.randint(15, 35)

        Y, X = np.ogrid[:img_size, :img_size]
        circle = (X - cx) ** 2 + (Y - cy) ** 2 <= r ** 2

        for c in range(3):
            images[i, :, :, c][circle] = np.random.uniform(0.6, 1.0)

        masks[i, :, :, 0][circle] = 1.0

    return images, masks


x_train, y_train = generate_synthetic_data(num_samples=1000)
x_test, y_test = generate_synthetic_data(num_samples=200)

# -----------------------------
# 2. U-Net Building Blocks
# -----------------------------
def double_conv_block(x, filters):
    """Two consecutive 3x3 convolutions with BN and ReLU."""
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


def encoder_block(x, filters):
    """Encoder: DoubleConv -> MaxPool. Returns skip and pooled."""
    skip = double_conv_block(x, filters)
    pooled = layers.MaxPool2D((2, 2))(skip)
    return skip, pooled


def decoder_block(x, skip, filters):
    """Decoder: UpConv -> Concat skip -> DoubleConv."""
    x = layers.Conv2DTranspose(filters, (2, 2), strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])  # Skip connection
    x = double_conv_block(x, filters)
    return x


# -----------------------------
# 3. Build U-Net Model
# -----------------------------
def build_unet(input_shape=(128, 128, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    # Bottleneck
    b = double_conv_block(p4, 1024)

    # Decoder
    d4 = decoder_block(b, s4, 512)
    d3 = decoder_block(d4, s3, 256)
    d2 = decoder_block(d3, s2, 128)
    d1 = decoder_block(d2, s1, 64)

    # Final 1x1 conv
    outputs = layers.Conv2D(num_classes, (1, 1), activation='sigmoid')(d1)

    model = models.Model(inputs, outputs)
    return model


model = build_unet()

# -----------------------------
# 4. Compile with Dice + BCE Loss
# -----------------------------
def dice_loss(y_true, y_pred, smooth=1.0):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(dice)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    bce = tf.reduce_mean(bce)
    dl = dice_loss(y_true, y_pred)
    return bce + dl


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=combined_loss,
    metrics=['accuracy'])

model.summary()

# -----------------------------
# 5. Train Model
# -----------------------------
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.1)

# -----------------------------
# 6. Visualization
# -----------------------------
# Predictions
preds = model.predict(x_test[:4])

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(4):
    axes[0, i].imshow(np.clip(x_test[i], 0, 1))
    axes[0, i].set_title("Input")
    axes[0, i].axis("off")

    axes[1, i].imshow(y_test[i, :, :, 0], cmap="gray")
    axes[1, i].set_title("Ground Truth")
    axes[1, i].axis("off")

    axes[2, i].imshow(preds[i, :, :, 0], cmap="gray")
    axes[2, i].set_title("Prediction")
    axes[2, i].axis("off")

plt.suptitle("U-Net Segmentation Results", fontsize=16)
plt.tight_layout()
plt.show()

# Training curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.show()